In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load your dataset
df = pd.read_csv("out.csv")  # columns: case_id, subject, case_explain, total_hearings, judgement, date

# Combine subject + case_explain for better context
df['combined'] = df['subject'] + " " + df['case_explain']

# Fit TF-IDF
vectorizer = TfidfVectorizer(stop_words='english')
X = vectorizer.fit_transform(df['combined'])

def find_similar_case(new_text):
    new_vec = vectorizer.transform([new_text])
    sims = cosine_similarity(new_vec, X)[0]
    idx = sims.argmax()
    return df.iloc[idx]  # return row of most similar case

# Example usage:
new_case = "A bank sues a borrower for not paying loan back"
match = find_similar_case(new_case)

print("Most similar case_id:", match['case_id'])
print("Judgement:", match['judgement'])
print("Total Hearings:", match['total_hearings'])
print("Date:", match['date'])

Most similar case_id: C004
Judgement: Borrower directed to repay loan in installments.
Total Hearings: 6
Date: 6/20/2024


In [3]:
from sentence_transformers import SentenceTransformer
import pandas as pd

# Load dataset
df = pd.read_csv("out.csv")

# Load embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Compute embeddings
df['embedding'] = df['case_explain'].apply(lambda x: model.encode(x))

# Save embeddings
import pickle
pickle.dump(df, open("cases_with_embeddings.pkl","wb"))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

X = df['case_explain']
y_judgement = df['judgement']              # classification
y_hearings = df['total_hearings']          # regression

# Judgement classifier
clf_judgement = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000)),
    ('rf', RandomForestClassifier())
])
clf_judgement.fit(X, y_judgement)

# Hearings regressor
from sklearn.ensemble import RandomForestRegressor
reg_hearings = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000)),
    ('rf', RandomForestRegressor())
])
reg_hearings.fit(X, y_hearings)

# Predict on new case
new_case = "Insurance claim after accident"
predicted_judgement = clf_judgement.predict([new_case])[0]
predicted_hearings = reg_hearings.predict([new_case])[0]

print(predicted_judgement, predicted_hearings)


Property divided equally between both parties. 5.2


In [4]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import pickle

# Load your precomputed embeddings
df = pickle.load(open("cases_with_embeddings.pkl","rb"))

def get_top3_cases(query_text):
    query_emb = model.encode(query_text)
    sims = cosine_similarity([query_emb], list(df['embedding']))[0]
    top_idx = np.argsort(sims)[::-1][:3]
    return df.iloc[top_idx][['case_id','subject','judgement','total_hearings','date','case_explain']]

# Example
query = "Insurance claim after road accident"
print(get_top3_cases(query))

  case_id          subject                                         judgement  \
1    C002   Accident Claim      Court orders ?5 lakh compensation to victim.   
2    C003  Contract Breach          Vendor liable to pay penalty for breach.   
3    C004     Loan Default  Borrower directed to repay loan in installments.   

   total_hearings        date  \
1               7  10/30/2022   
2               4   1/11/2024   
3               6   6/20/2024   

                                        case_explain  
1  Compensation case filed by victim of road acci...  
2  Software company sued vendor for breach of con...  
3  Bank filed a case for recovery of loan from bo...  


In [1]:
import PyPDF2
from transformers import pipeline

# Step 1: Load summarization model
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")
# summarizer = pipeline("summarization", model="sshleifer/distilbart-cnn-12-6")

# Step 2: Read your PDF
pdf_path = "cybercrime_complaint.pdf"

text = ""
with open(pdf_path, "rb") as file:
    reader = PyPDF2.PdfReader(file)
    for page in reader.pages:
        extracted = page.extract_text()
        if extracted:
            text += extracted + "\n"

# Step 3: Handle long text (very important ⚠️)
def split_text(text, max_chunk=500):
    sentences = text.split(". ")
    chunks = []
    chunk = ""

    for sentence in sentences:
        if len(chunk) + len(sentence) <= max_chunk:
            chunk += sentence + ". "
        else:
            chunks.append(chunk)
            chunk = sentence + ". "

    if chunk:
        chunks.append(chunk)

    return chunks

chunks = split_text(text)

# Step 4: Summarize each chunk
summary_list = []

for chunk in chunks:
    summary = summarizer(chunk, max_length=80, min_length=25, do_sample=False)
    summary_list.append(summary[0]['summary_text'])

# Step 5: Combine final summary
final_summary = " ".join(summary_list)

print("\n===== FINAL SUMMARY =====\n")
print(final_summary)

c:\Users\harih\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



===== FINAL SUMMARY =====

Hariharan R, residing at Vellore, Tamil Nadu, would like to bring to your kind notice that I have                been a victim of cyber fraud on March 12, 2026. On the mentioned date, I received a phone call from an unknown person claiming to be a bank official. The caller informed me about a supposed issue with my bank account. He requested me to share my OTP and debit card details for verification purposes. Believing the caller to be genuine, I provided the requested details.Shortly after the call, I noticed multiple unauthorized transactions. I immediately contacted my bank and blocked my card. I request you to kindly register my complaint and initiate an investigation into this matter. However, the transactions had already been processed. I have attached screenshots of transaction messages, call logs, and bank statements as evidence.


In [1]:
pip list

Package                      Version
---------------------------- ---------------
absl-py                      2.3.1
aiohappyeyeballs             2.6.1
aiohttp                      3.13.3
aiosignal                    1.4.0
annotated-doc                0.0.4
annotated-types              0.7.0
anyio                        4.12.1
APScheduler                  3.11.2
asttokens                    3.0.1
astunparse                   1.6.3
attrs                        25.4.0
audioread                    3.1.0
bcrypt                       5.0.0
beautifulsoup4               4.14.3
bitarray                     3.8.0
blinker                      1.9.0
certifi                      2026.1.4
cffi                         2.0.0
charset-normalizer           3.4.4
ckzg                         2.1.7
click                        8.1.8
colorama                     0.4.6
comm                         0.2.3
contourpy                    1.1.0
cycler                       0.12.1
cytoolz                      1.1.0

In [5]:
# Generate 1000 synthetic court case records and save as CSV

import pandas as pd
import random

case_types = ["civil", "criminal", "cyber"]
court_levels = ["district", "high", "supreme"]

data = []

for _ in range(1000):
    case_type = random.choice(case_types)
    complexity = random.randint(1, 5)
    witnesses = random.randint(1, 20)
    evidence_docs = random.randint(5, 100)
    lawyer_exp = random.randint(1, 25)
    court = random.choice(court_levels)
    adjournments = random.randint(0, 10)

    # Generate target with some logic
    base = complexity * 5 + witnesses * 1.5 + evidence_docs * 0.2 + adjournments * 2
    if case_type == "criminal":
        base += 10
    elif case_type == "cyber":
        base += 5

    if court == "high":
        base += 5
    elif court == "supreme":
        base += 10

    total_hearings = int(base + random.randint(-5, 5))

    data.append([
        case_type, complexity, witnesses, evidence_docs,
        lawyer_exp, court, adjournments, total_hearings
    ])

df = pd.DataFrame(data, columns=[
    "case_type", "case_complexity", "num_witnesses",
    "num_evidence_docs", "lawyer_experience",
    "court_level", "previous_adjournments", "total_hearings"
])

file_path = "court_cases_1000.csv"
df.to_csv(file_path, index=False)

file_path

'court_cases_1000.csv'

In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder

# Load dataset
df = pd.read_csv("court_cases_1000.csv")

# Encode categorical columns
le_case = LabelEncoder()
le_court = LabelEncoder()

df["case_type"] = le_case.fit_transform(df["case_type"])
df["court_level"] = le_court.fit_transform(df["court_level"])

# Features & target
X = df.drop("total_hearings", axis=1)
y = df["total_hearings"]

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Train model
model = RandomForestRegressor(n_estimators=100)
model.fit(X_train, y_train)

print("Model trained successfully!")

Model trained successfully!


In [10]:
import joblib

# Save model
joblib.dump(model, "hearing_model.pkl")

# Save encoders also (VERY IMPORTANT ⚠️)
joblib.dump(le_case, "case_encoder.pkl")
joblib.dump(le_court, "court_encoder.pkl")

print("Model and encoders saved!")

Model and encoders saved!


In [11]:
import joblib
import pandas as pd

# Load model & encoders
model = joblib.load("hearing_model.pkl")
le_case = joblib.load("case_encoder.pkl")
le_court = joblib.load("court_encoder.pkl")

print("Model loaded!")

Model loaded!


In [ ]:
# Example new case
new_case = {
    "case_type": "cyber",
    "case_complexity": 4,
    "num_witnesses": 5,
    "num_evidence_docs": 35,
    "lawyer_experience": 10,
    "court_level": "high",
    "previous_adjournments": 3
}

# Encode input
new_case["case_type"] = le_case.transform([new_case["case_type"]])[0]
new_case["court_level"] = le_court.transform([new_case["court_level"]])[0]

import pandas as pd
new_df = pd.DataFrame([new_case])

# Predict
prediction = model.predict(new_df)

print("Predicted Total Hearings:", round(prediction[0]))

Predicted Total Hearings: 54


In [13]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

# Predict on test data
y_pred = model.predict(X_test)

# Metrics
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("R2 Score (Accuracy):", r2)
print("MAE:", mae)
print("RMSE:", rmse)

R2 Score (Accuracy): 0.8643478746072766
MAE: 4.6428
RMSE: 5.654125308126802
